# Multi-Agent Orchestration for RAG Systems
# Step 2. Specialized Agents for Orchestrated RAG

*Assignee: Alla* - *Review:*

This notebook implements the specialized-agent layer for a modular RAG system. It defines role-based agents with a shared `AgentState` contract and keeps retriever artifacts reusable for later orchestration experiments.

The design is intentionally decoupled from routing, so external orchestrators can apply strategies such as Parallel+Fusion, Sequential Waterfall, or Confidence-Based Routing without changing agent internals.



In [43]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers


In [44]:
import os
import re
import json
import pickle
import random
import pathlib
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import nltk
from langdetect import detect

from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')



Setup complete.


In [45]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Setup

Requirements:
- Add `HF_TOKEN` to Colab Secrets (https://huggingface.co/settings/tokens).
- Ensure Google Drive contains:
  - `/content/drive/MyDrive/Adv_GenAI/benchmark`
  - `/content/drive/MyDrive/Adv_GenAI/storage`

Scope selection:
- `EVAL_SCOPE = 'full_corpus'` or `EVAL_SCOPE = 'subsample'`
- For orchestration comparison, `full_corpus` is recommended.



In [46]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

# Choose scope: 'subsample' or 'full_corpus'
EVAL_SCOPE = 'full_corpus'
assert EVAL_SCOPE in {'subsample', 'full_corpus'}
print('EVAL_SCOPE =', EVAL_SCOPE)

if EVAL_SCOPE == 'subsample':
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
    if not PATH_BM25_PICKLE.exists():
        PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval/fixed_size_chunk/bm25_retriever.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'
else:
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/full_corpus/retrieval/fixed_size_chunk/bm25_retriever_full.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/full_corpus/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/full_corpus/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/full_corpus/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

required = [PATH_BM25_PICKLE, PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required retrieval artifacts found.')



PROJECT_ROOT = /content/drive/MyDrive/Adv_GenAI
EVAL_SCOPE = full_corpus
All required retrieval artifacts found.


## 2. Multi-Agent Architecture (Specialized Roles)

This is the main implementation section.

Implemented roles:
- Query Understanding Agent
- Retriever Agents (`BM25`, `Dense`, `GraphRAG`)
- Fusion Agent (merge + deduplicate)
- Re-Ranker Agent
- Answer Synthesizer Agent
- Critic Agent (grounding check + re-retrieval trigger)

All agents read/write a shared `AgentState`, making routing strategies interchangeable.



## 2.1 Retrieval Component: BM25 Adapter

Loads BM25 artifacts with a compatibility wrapper and exposes a stable `search(query, top_k)` interface across artifact variants.

**Retriever Compatibility Note**
Some BM25 artifacts were serialized with custom classes (e.g., `BilingualBM25` or `QEBM25`). During `pickle.load(...)`, Python must resolve those class definitions to reconstruct the object, even if we later access the retriever only through `BM25RetrieverAdapter`.

For this reason, these class definitions are intentionally kept in the notebook as deserialization shims for cross-version artifact compatibility.




In [47]:
# Robust BM25 loader for both subsample and full-corpus pickle formats
class BilingualBM25:
    """Compatibility class for notebook pickles."""

    def _rank_lang(self, q: str, lang: str, k: int):
        # Subsample-style object: self.bm25 + self.docs_by_lang
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def _get_docs_with_scores(self, ret, qq, top_k):
        # Full-corpus-style object: self.retrievers
        if hasattr(ret, 'get_relevant_documents_with_scores'):
            try:
                return ret.get_relevant_documents_with_scores(qq, k=top_k)
            except Exception:
                pass

        if hasattr(ret, 'vectorizer') and hasattr(ret, 'docs'):
            try:
                toks = qq.lower().split()
                scores = ret.vectorizer.get_scores(toks)
                ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
                return [(ret.docs[idx], float(score)) for idx, score in ranked]
            except Exception:
                pass

        if hasattr(ret, 'invoke'):
            try:
                old_k = getattr(ret, 'k', None)
                if old_k is not None:
                    ret.k = top_k
                docs = ret.invoke(qq)
                if old_k is not None:
                    ret.k = old_k
                return [(d, d.metadata.get('score', 0.0)) for d in docs[:top_k]]
            except Exception:
                pass

        return []

    def search(self, query: str, top_k: int = 100):
        # Route to correct behavior based on available attributes
        if hasattr(self, 'bm25') and hasattr(self, 'docs_by_lang'):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)
            for lang in ('en', 'de'):
                q_lang = translator.translate(query, lang) if translator and lang != src else query
                bag.extend(self._rank_lang(q_lang, lang, top_k))

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d
            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        if hasattr(self, 'retrievers') and isinstance(self.retrievers, dict):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)

            for lang, ret in self.retrievers.items():
                qq = translator.translate(query, lang) if translator and lang != src else query
                docs_with_scores = self._get_docs_with_scores(ret, qq, top_k)
                for doc, score in docs_with_scores:
                    doc.metadata['bm25_score'] = float(score)
                    bag.append(doc)

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid is None:
                    continue
                if uid not in best or d.metadata.get('bm25_score', -1e9) > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d

            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        raise AttributeError('Unsupported BilingualBM25 object format.')

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        if hasattr(self, 'base'):
            expanded = self._expand_query(query, self.base)
            return self.base.search(expanded, top_k)
        raise AttributeError('QEBM25 object missing base retriever.')

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_raw = pickle.load(f)

class BM25RetrieverAdapter:
    def __init__(self, obj):
        self.obj = obj

    def search(self, query: str, top_k: int = 100):
        # Primary path
        if hasattr(self.obj, 'search'):
            try:
                return self.obj.search(query, top_k=top_k)
            except TypeError:
                return self.obj.search(query, k=top_k)

        # LangChain retriever fallback
        if hasattr(self.obj, 'invoke'):
            old_k = getattr(self.obj, 'k', None)
            if old_k is not None:
                self.obj.k = top_k
            docs = self.obj.invoke(query)
            if old_k is not None:
                self.obj.k = old_k
            for rank, d in enumerate(docs, start=1):
                if hasattr(d, 'metadata'):
                    d.metadata.setdefault('bm25_score', float(top_k - rank))
            return docs[:top_k]

        raise AttributeError(f'Unsupported BM25 object type: {type(self.obj)}')

bm25_retriever = BM25RetrieverAdapter(bm25_raw)
print('BM25 loaded:', type(bm25_raw), '-> adapter ready')


BM25 loaded: <class '__main__.BilingualBM25'> -> adapter ready


### Explanation: BM25 compatibility loader + adapter

This cell handles different BM25 artifact formats from previous runs.

What it ensures:
1. Old pickles can be deserialized safely (`BilingualBM25`, `QEBM25`).
2. Different object styles still expose one interface: `search(query, top_k)`.
3. Result docs carry comparable metadata (e.g., `bm25_score`).

`BM25RetrieverAdapter` is the final stable wrapper used by the pipeline.

## 2.2 Retrieval Component: Dense Retriever

Initializes the multilingual E5 + Chroma retriever and returns dense candidates through the same interface used by other retrievers.



In [48]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Dense retriever ready.


### Explanation: `DenseRetriever`

Dense retrieval uses multilingual embeddings and vector search.

How it works:
1. Prefix query as `query: ...` for E5-style embedding format.
2. Run Chroma similarity search.
3. Convert distances to similarity-style score (`dense_score = 1 - dist`).
4. Return ranked semantic candidates.

This helps with paraphrases and semantic meaning beyond exact keyword match.

## 2.3 Retrieval Component: GraphRAG Retriever

Loads GraphRAG resources and returns graph-informed candidates by selecting relevant communities and ranking chunks within them.



In [49]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)

        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = 'C1', k_comms: int = 24, top_k: int = 100):
        L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100, k_comms: int = 48):
        return self.retrieve(query=query, level='C1', k_comms=k_comms, top_k=top_k)

graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GraphRAG retriever ready.


### Explanation: `GraphRAGRetriever`

This retriever is graph-guided + semantic.

Step-by-step:
1. Load community embeddings and chunk mappings.
2. Encode query embedding.
3. Select top communities (`k_comms`).
4. Gather chunk candidates from those communities.
5. Score candidates by embedding similarity.
6. Return top `top_k` docs with `grag_score` metadata.

Good for relationship/context-heavy questions.

In [50]:
# Shared utilities used by Step 2 agents
from collections import defaultdict
from typing import Any

def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    # Balanced defaults: Dense leads (better semantic coverage),
    # BM25 equal (strong for factoid), Graph gets a fair share.
    weights = weights or {'bm25': 1.0, 'dense': 1.2, 'graph': 0.8}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _token_set(text: str):
    text = (text or '').lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(t for t in text.split() if t)

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

print('Shared Step 2 utilities ready.')


Shared Step 2 utilities ready.


### Explanation: Shared utilities

These helpers are used by multiple agents:
- `_uid(...)`: extract stable document ID.
- `_safe_unique(...)`: remove duplicates / invalid IDs.
- `_rrf_fuse(...)`: weighted reciprocal-rank fusion.
- `_token_set(...)`: normalization + tokenization.
- `_overlap_rerank(...)`: simple fallback reranker.

They keep behavior consistent across pipeline stages.

In [51]:
# Multi-agent role contracts
from dataclasses import dataclass, field
from typing import Dict, List, Any

@dataclass
class AgentState:
    query: str
    normalized_query: str = ''
    query_type: str = 'mixed'
    query_hints: Dict[str, float] = field(default_factory=dict)
    retrieval_by_agent: Dict[str, List[Any]] = field(default_factory=dict)
    fused_docs: List[Any] = field(default_factory=list)
    reranked_docs: List[Any] = field(default_factory=list)
    final_answer: str = ''
    evidence_ids: List[str] = field(default_factory=list)
    critic_ok: bool = False
    critic_feedback: str = ''
    needs_reretrieval: bool = False

class BaseAgent:
    name = 'base'
    def run(self, state: AgentState, **kwargs) -> AgentState:
        raise NotImplementedError

print('Agent contracts ready.')



Agent contracts ready.


### Explanation: `AgentState` and base contract

`AgentState` is the shared memory object passed through all agents.

Main fields:
- `query`, `normalized_query`, `query_type`
- `query_hints` (weights + hints)
- `retrieval_by_agent`, `fused_docs`, `reranked_docs`
- `final_answer`, `evidence_ids`
- `critic_ok`, `critic_feedback`, `needs_reretrieval`

`BaseAgent.run(...)` gives one consistent interface for all agent components.

In [52]:
# 1) Query Understanding Agent
import re as _re

class QueryUnderstandingAgent(BaseAgent):
    name = 'query_understanding'

    # Matches "who was/is/were [role]" or "who served as" patterns
    _WHO_ROLE_RE = _re.compile(r'^who\b', _re.I)  # any question starting with 'who'
    # Matches a 4-digit year anywhere in the query
    _YEAR_RE = _re.compile(r'\b(1\d{3}|20\d{2})\b')

    def run(self, state: AgentState, **kwargs) -> AgentState:
        q = (state.query or '').strip()
        q_low = q.lower()

        state.normalized_query = ' '.join(q.split())

        graph_signals = {'relationship', 'connected', 'connection', 'connections', 'dependency', 'impact', 'between'}
        keyword_signals = {'exactly', 'define', 'list', 'when', 'where', 'who'}

        gr_score  = sum(1 for t in graph_signals  if t in q_low)
        kw_score  = sum(1 for t in keyword_signals if t in q_low)

        has_year        = bool(self._YEAR_RE.search(q))
        who_role_query  = bool(self._WHO_ROLE_RE.match(q))
        factual_starts  = (
            'when ', 'where ', 'which ',
            'what year', 'what date',
            'what is ', 'what are ', 'what was ', 'what were ',
            'what does ', 'what do ', 'what did ',
        )
        factual_query   = q_low.startswith(factual_starts)

        # "Who was/is [role]" + year → entity-temporal: graph community search
        # is best for finding person-role-time triples.
        if who_role_query and has_year:
            q_type = 'entity_temporal'
        elif who_role_query:
            q_type = 'entity'
        elif gr_score >= 1:
            q_type = 'graph'
        elif factual_query or kw_score >= 1:
            q_type = 'keyword'
        elif len(q.split()) >= 7:
            q_type = 'semantic'
        else:
            q_type = 'mixed'

        state.query_type = q_type

        # Store year in state so downstream agents can use it
        m = self._YEAR_RE.search(q)
        state.query_hints['_year'] = int(m.group()) if m else None

        # Fusion weights by type
        if q_type == 'entity_temporal':
            # Graph captures person-role-org community structure best;
            # Dense for semantic similarity; BM25 suppressed (year matches noise).
            state.query_hints.update({'bm25': 0.7, 'dense': 1.1, 'graph': 1.4})
        elif q_type == 'entity':
            state.query_hints.update({'bm25': 0.9, 'dense': 1.1, 'graph': 1.3})
        elif q_type == 'keyword':
            state.query_hints.update({'bm25': 1.3, 'dense': 1.1, 'graph': 0.7})
        elif q_type == 'semantic':
            state.query_hints.update({'bm25': 0.8, 'dense': 1.4, 'graph': 0.9})
        elif q_type == 'graph':
            state.query_hints.update({'bm25': 0.8, 'dense': 1.0, 'graph': 1.4})
        else:  # mixed
            state.query_hints.update({'bm25': 1.0, 'dense': 1.2, 'graph': 0.8})

        return state

print('Query Understanding Agent ready.')


Query Understanding Agent ready.


### Explanation: `QueryUnderstandingAgent`

Purpose: interpret query type and set retrieval weights.

Key actions:
1. Normalize query text.
2. Detect signals (factoid/semantic/graph/entity-temporal).
3. Assign `state.query_type`.
4. Store year hint in `state.query_hints['_year']` when present.
5. Set dynamic retriever weights (`bm25`, `dense`, `graph`).

Why useful: different query types benefit from different retrieval emphasis.

In [53]:
# 2) Retriever Agents (BM25, Dense, GraphRAG)
class BM25RetrieverAgent(BaseAgent):
    name = 'bm25_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['bm25'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class DenseRetrieverAgent(BaseAgent):
    name = 'dense_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['dense'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class GraphRetrieverAgent(BaseAgent):
    name = 'graph_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['graph'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k, k_comms=48))
        return state

print('Retriever Agents ready.')



Retriever Agents ready.


### Explanation: Retriever Agents (`BM25`, `Dense`, `Graph`)

These wrappers standardize retrieval calls.

What each `run(...)` does:
1. Read `state.normalized_query`.
2. Call retriever with `top_k`.
3. Clean results via `_safe_unique`.
4. Store per-agent outputs in `state.retrieval_by_agent`.

This makes downstream fusion independent of retriever internals.

In [54]:
# 3) Fusion Agent
class FusionAgent(BaseAgent):
    name = 'fusion'

    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        runs = {
            'bm25': state.retrieval_by_agent.get('bm25', []),
            'dense': state.retrieval_by_agent.get('dense', []),
            'graph': state.retrieval_by_agent.get('graph', []),
        }
        fused = _rrf_fuse(runs, weights=state.query_hints or None)
        state.fused_docs = _safe_unique(fused)[:top_k]
        return state

print('Fusion Agent ready.')



Fusion Agent ready.


### Explanation: `FusionAgent`

Purpose: merge outputs from BM25, Dense, and Graph retrievers into one ranked list.

How fusion works:
1. Collect runs from `state.retrieval_by_agent`.
2. Apply weighted RRF (`_rrf_fuse`) using `state.query_hints` when available.
3. Remove duplicates/invalid IDs (`_safe_unique`).
4. Keep top fused documents in `state.fused_docs`.

RRF merges ranked outputs by document IDs (not by merging text chunks).

In [55]:
# 4) Re-Ranker Agent — CrossEncoder with overlap fallback
class ReRankerAgent(BaseAgent):
    name = 'reranker'
    _ce_model = None  # class-level cache; loaded once per session

    @classmethod
    def _get_cross_encoder(cls):
        if cls._ce_model is None:
            print('[ReRanker] Loading CrossEncoder (ms-marco-MiniLM-L-6-v2)...')
            cls._ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            print('[ReRanker] CrossEncoder ready.')
        return cls._ce_model

    def run(self, state: AgentState, top_k: int = 10, **kwargs) -> AgentState:
        docs = state.fused_docs
        if not docs:
            state.reranked_docs = []
            return state

        query = state.normalized_query
        # Limit cross-encoder to top-50 fused candidates to keep latency reasonable
        candidates = docs[:50]

        try:
            model = self._get_cross_encoder()
            pairs = [
                (query, (d.metadata.get('original_text') or d.page_content or '').strip()[:512])
                for d in candidates
            ]
            scores = model.predict(pairs)
            ranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)
            for d, score in ranked:
                d.metadata['rerank_score'] = float(score)
            state.reranked_docs = [d for d, _ in ranked[:top_k]]
        except Exception as e:
            # Overlap rerank as fallback so the pipeline never silently breaks
            print(f'[ReRanker] CrossEncoder failed ({e}), using overlap fallback.')
            state.reranked_docs = _overlap_rerank(docs, query, top_k=top_k)
            for d in state.reranked_docs:
                d.metadata.setdefault('rerank_score', 0.0)

        return state

print('Re-Ranker Agent ready (CrossEncoder + overlap fallback).')


Re-Ranker Agent ready (CrossEncoder + overlap fallback).


### Explanation: `ReRankerAgent`

Purpose: improve ranking quality of top fused candidates.

Step-by-step:
1. Start from `state.fused_docs`.
2. Limit candidates to top 50 for speed.
3. Build `(query, doc_text)` pairs.
4. CrossEncoder predicts relevance scores.
5. Sort by score descending.
6. Keep top `top_k` in `state.reranked_docs`.
7. If CrossEncoder fails, fallback to overlap rerank.

It reranks document order; it does not change document content.

In [56]:
# 5) Answer Synthesizer Agent — temporal-aware extractive scoring
class AnswerSynthesizerAgent(BaseAgent):
    name = 'answer_synthesizer'

    def _build_context(self, docs: List[Any], max_docs: int = 5) -> str:
        parts = []
        for d in docs[:max_docs]:
            txt = (d.metadata.get('original_text') or d.page_content or '').strip()
            if txt:
                parts.append(txt)
        return "\n\n".join(parts)

    def run(self, state: AgentState, **kwargs) -> AgentState:
        ctx = self._build_context(state.reranked_docs or state.fused_docs)
        q = state.normalized_query
        year = state.query_hints.get('_year')  # e.g. 2003; None if no year in query

        if not ctx:
            state.final_answer = 'No supporting context was retrieved.'
            state.evidence_ids = []
            return state

        sents = re.split(r'(?<=[.!?])\s+', ctx)
        q_terms = set(_token_set(q)) - STOP_EN - STOP_DE
        scored = []
        for s in sents:
            s = s.strip()
            if len(s) < 20:
                continue
            st = set(_token_set(s))
            # Base score: normalised query-term overlap * raw match count
            base = len(q_terms & st) / max(len(st), 1) * len(q_terms & st)
            # Temporal bonus: sentence contains the query year or adjacent year
            if year and re.search(r'\b' + str(year) + r'\b', s):
                base *= 2.5
            elif year and any(
                re.search(r'\b' + str(y) + r'\b', s)
                for y in [year - 1, year + 1, year - 2, year + 2]
            ):
                base *= 1.4
            scored.append((base, s))

        scored.sort(key=lambda x: x[0], reverse=True)

        best = []
        seen_terms: set = set()
        for score, s in scored:
            s_terms = set(_token_set(s))
            if seen_terms and len(s_terms & seen_terms) / max(len(s_terms), 1) > 0.6:
                continue
            best.append(s)
            seen_terms |= s_terms
            if len(best) >= 3:
                break

        # If the top-scoring sentence has a zero score, no query terms matched at all —
        # report honestly rather than returning misleading extractive content.
        if not best or scored[0][0] == 0:
            state.final_answer = (
                f'The retrieved context does not contain information about the query'
                + (f' from {year}' if year else '') + '.'
            )
        else:
            state.final_answer = ' '.join(best)

        state.evidence_ids = [_uid(d) for d in (state.reranked_docs or state.fused_docs)[:5] if _uid(d) is not None]
        return state

print('Answer Synthesizer Agent ready.')


Answer Synthesizer Agent ready.


### Explanation: `AnswerSynthesizerAgent`

Purpose: produce an answer from retrieved evidence (mainly extractive, not free-form generation).

Step-by-step:
1. Build context from best docs (`reranked_docs` first, else `fused_docs`).
2. Split context into candidate sentences.
3. Score each sentence by query-term overlap.
4. Add temporal bonus if sentence matches query year.
5. Keep top diverse sentences.
6. Join selected sentences into `state.final_answer`.
7. Save supporting IDs in `state.evidence_ids`.

This keeps answers grounded in retrieved evidence.

In [57]:
# 6) Critic Agent — temporal coherence check for entity_temporal queries
import re as _re2

class CriticAgent(BaseAgent):
    name = 'critic'
    _YEAR_RE = _re2.compile(r'\b(1\d{3}|20\d{2})\b')

    def _temporal_coherent(self, answer: str, query_year: int, window: int = 10) -> bool:
        """Answer must mention at least one year within ±window of the query year.
        Catches answers that are grounded but reference the wrong decade entirely."""
        found = [int(m) for m in self._YEAR_RE.findall(answer)]
        return any(abs(y - query_year) <= window for y in found)

    def run(self, state: AgentState, min_support_overlap: float = 0.45, **kwargs) -> AgentState:
        answer_terms = _token_set(state.final_answer) - STOP_EN - STOP_DE
        if not answer_terms:
            state.critic_ok = False
            state.needs_reretrieval = True
            state.critic_feedback = 'Answer is empty.'
            return state

        support_docs = state.reranked_docs or state.fused_docs
        if not support_docs:
            state.critic_ok = False
            state.needs_reretrieval = True
            state.critic_feedback = 'No support documents available.'
            return state

        support_text = ' '.join(
            (d.metadata.get('original_text') or d.page_content or '') for d in support_docs[:5]
        )
        support_terms = _token_set(support_text) - STOP_EN - STOP_DE
        overlap = len(answer_terms & support_terms) / max(len(answer_terms), 1)

        per_doc_max = 0.0
        for d in support_docs[:5]:
            dt = _token_set(d.metadata.get('original_text') or d.page_content or '') - STOP_EN - STOP_DE
            per_doc_max = max(per_doc_max, len(answer_terms & dt) / max(len(answer_terms), 1))

        grounded = overlap >= min_support_overlap and per_doc_max >= 0.25

        # Temporal coherence: for entity_temporal queries the answer must contain
        # a year within ±10 of the query year.  This catches answers that are
        # lexically grounded (contain 'president', 'ETH') but reference the wrong
        # decade — e.g. returning info about the 2019 president for a 2003 query.
        query_year = state.query_hints.get('_year')
        temporal_ok = True
        temporal_msg = ''
        if state.query_type == 'entity_temporal' and query_year:
            temporal_ok = self._temporal_coherent(state.final_answer, query_year, window=10)
            temporal_msg = (
                f' Temporal check (target={query_year}±10): '
                + ('PASS' if temporal_ok else 'FAIL — answer references wrong time period') + '.'
            )

        state.critic_ok = grounded and temporal_ok
        state.needs_reretrieval = not state.critic_ok
        state.critic_feedback = (
            f'Global overlap={overlap:.3f}, best-doc overlap={per_doc_max:.3f}; '
            f'threshold={min_support_overlap:.2f}/0.25.'
            + temporal_msg
            + (' Grounded.' if state.critic_ok else ' Potentially ungrounded: trigger re-retrieval.')
        )
        return state

print('Critic Agent ready.')


Critic Agent ready.


### Explanation: `CriticAgent`

Purpose: verify whether the generated answer is supported by retrieved evidence.

How it critiques:
1. Build `answer_terms` from answer text (normalized tokens, stopwords removed).
2. Build `support_terms` from top support docs.
3. Compute overlap ratio between answer and support terms.
4. Check thresholds (`min_support_overlap`, `per_doc_max >= 0.25`).
5. For temporal queries, check year consistency (`temporal_ok`).
6. Set `state.critic_ok` and `state.needs_reretrieval`.

Important: `min_support_overlap` is a code heuristic (default `0.45`), not a benchmark-defined value.

In [58]:
# 7) Orchestrator-ready linear execution (can be replaced by external orchestrator)
class MultiAgentPipeline:
    def __init__(self, use_graph: bool = True):
        self.use_graph = use_graph
        self.query_agent = QueryUnderstandingAgent()
        self.bm25_agent = BM25RetrieverAgent(bm25_retriever)
        self.dense_agent = DenseRetrieverAgent(dense_retriever)
        self.graph_agent = GraphRetrieverAgent(graph_retriever)
        self.fusion_agent = FusionAgent()
        self.reranker_agent = ReRankerAgent()
        self.answer_agent = AnswerSynthesizerAgent()
        self.critic_agent = CriticAgent()

    def _safe_retrieval(self, agent, state: AgentState, key: str, top_k: int):
        try:
            return agent.run(state, top_k=top_k)
        except Exception as e:
            state.retrieval_by_agent[key] = []
            state.critic_feedback = (
                (state.critic_feedback + ' ') if state.critic_feedback else ''
            ) + f'{key} retrieval failed: {e}'
            return state

    def _retrieve_all(self, state: AgentState, retrieve_k: int):
        state = self._safe_retrieval(self.bm25_agent, state, 'bm25', retrieve_k)
        state = self._safe_retrieval(self.dense_agent, state, 'dense', retrieve_k)
        if self.use_graph:
            state = self._safe_retrieval(self.graph_agent, state, 'graph', retrieve_k)
        else:
            state.retrieval_by_agent['graph'] = []
        return state

    def run(self, query: str, retrieve_k: int = 50, top_k: int = 5, retry_once: bool = True):
        state = AgentState(query=query)

        # Step A: understanding
        state = self.query_agent.run(state)

        # Step B: retrieval (all three retrievers, isolated failures)
        state = self._retrieve_all(state, retrieve_k)

        # Step C: fusion -> rerank -> synthesis -> critique
        state = self.fusion_agent.run(state, top_k=retrieve_k)
        state = self.reranker_agent.run(state, top_k=top_k)
        state = self.answer_agent.run(state)
        state = self.critic_agent.run(state)

        # Retry with boosted weights if critic flags the answer as ungrounded
        if retry_once and state.needs_reretrieval:
            print('[Pipeline] Critic triggered re-retrieval...')
            boosted = dict(state.query_hints)
            boosted['bm25'] = boosted.get('bm25', 1.0) + 0.3
            boosted['dense'] = boosted.get('dense', 1.0) + 0.3
            state.query_hints = boosted
            state = self._retrieve_all(state, retrieve_k)
            state = self.fusion_agent.run(state, top_k=retrieve_k)
            state = self.reranker_agent.run(state, top_k=top_k)
            state = self.answer_agent.run(state)
            state = self.critic_agent.run(state)

        # If the critic still disagrees after all attempts, be honest rather
        # than returning a plausible-sounding but wrong extractive answer.
        if not state.critic_ok:
            year_hint = state.query_hints.get('_year')
            year_str  = f' from {year_hint}' if year_hint else ''
            state.final_answer = (
                f'The available corpus does not contain sufficient evidence to '
                f'reliably answer this query{year_str}. Retrieved context covers '
                f'related topics but does not directly address the question.'
            )
        return state

pipeline = MultiAgentPipeline(use_graph=True)
print('Multi-agent pipeline ready. Graph enabled:', pipeline.use_graph)


Multi-agent pipeline ready. Graph enabled: True


### Explanation: `MultiAgentPipeline` (sequential orchestration)

This is the core execution flow.

Flow:
1. `QueryUnderstandingAgent` classifies query + sets weights.
2. Retrieve from BM25/Dense/Graph agents.
3. `FusionAgent` merges rankings (RRF + dedupe).
4. `ReRankerAgent` improves top ordering.
5. `AnswerSynthesizerAgent` composes answer from evidence.
6. `CriticAgent` checks grounding/temporal consistency.
7. If critic fails, run one retry with boosted retrieval weights.
8. If still unsupported, return an honest fallback answer.

This is a sequential pipeline strategy and can be compared to other strategies in Step 3.

In [67]:
# Example usage
sample_query = 'Who were the rectors of ETH between 2017 and 2022?'
out = pipeline.run(sample_query, retrieve_k=20, top_k=5, retry_once=True)

print('Query type:', out.query_type)
print('Query hints:', {k: v for k, v in out.query_hints.items() if not str(k).startswith('_')})
print('Critic OK:', out.critic_ok)
print('Critic feedback:', out.critic_feedback)
print('Evidence IDs:', out.evidence_ids[:5])
print('\nFinal answer:\n', out.final_answer)


Query type: entity_temporal
Query hints: {'bm25': 0.7, 'dense': 1.1, 'graph': 1.4}
Critic OK: True
Critic feedback: Global overlap=1.000, best-doc overlap=0.561; threshold=0.45/0.25. Temporal check (target=2017±10): PASS. Grounded.
Evidence IDs: ['73afc89edd471ff98176f33babf28b62dccf4ac6_fixed_2', '00859327fafc62821b53b9ad083e2a244c1b4470_fixed_3', '20aed32900c864b6d99d046fa3706c8dec5194d6_fixed_0', 'b7471c1deec222c6c0529cfaf9285b9c075c5096_fixed_1', 'ce4529dd92b5b28f5b642e7a04f73206e38024f0_fixed_1']

Final answer:
 the [vice rector for continuing education](https://ethz.ch/en/the-eth-zurich/organisation/vice-rectors.html) assists the rector in the area of continuing education for those with an academic background at eth zurich. reaching critical mass of women at eth: /equal-opportunities/strategie-und-zahlen/frauen-an-der-eth/frauenstreik-eth/promoting-women.html) **sarah springman** professor of geotechnical engineering at eth zurich and rector of eth zurich until the end of january

In [68]:
# Retriever debug view: inspect top results before and after fusion

def _preview_docs(name, docs, n=3, max_chars=220):
    print(f"\n[{name}] top {min(n, len(docs))} / {len(docs)}")
    for i, d in enumerate(docs[:n], start=1):
        uid = _uid(d)
        text = (d.metadata.get('original_text') or d.page_content or '').replace('\n', ' ').strip()
        text = text[:max_chars] + ('...' if len(text) > max_chars else '')
        score_keys = [k for k in ('bm25_score', 'dense_score', 'grag_score', 'fused_score', 'rerank_score') if k in d.metadata]
        score_str = ', '.join(f"{k}={d.metadata.get(k):.4f}" for k in score_keys)
        print(f"{i}. id={uid} | {score_str}")
        print(f"   {text}")


def debug_query(query, retrieve_k=50, top_k=5, include_graph=True):
    print('Query:', query)
    print('include_graph:', include_graph)

    # Individual retrievers
    bm = _safe_unique(bm25_retriever.search(query, top_k=retrieve_k))
    de = _safe_unique(dense_retriever.search(query, top_k=retrieve_k))
    gr = _safe_unique(graph_retriever.search(query, top_k=retrieve_k)) if include_graph else []

    _preview_docs('BM25', bm)
    _preview_docs('Dense', de)
    if include_graph:
        _preview_docs('GraphRAG', gr)

    # Full pipeline outputs
    dbg_pipeline = MultiAgentPipeline(use_graph=include_graph)
    out = dbg_pipeline.run(query, retrieve_k=retrieve_k, top_k=top_k, retry_once=True)
    _preview_docs('Fused', out.fused_docs)
    _preview_docs('Re-ranked', out.reranked_docs)

    print('\nQuery type:', out.query_type)
    print('Query hints:', out.query_hints)
    print('Critic OK:', out.critic_ok)
    print('Critic feedback:', out.critic_feedback)
    print('Evidence IDs:', out.evidence_ids)
    print('\nFinal answer:\n', out.final_answer)

# Example debug call:
debug_query('Who were the rectors of ETH between 2017 and 2022?', retrieve_k=20, top_k=5, include_graph=True)


Query: Who were the rectors of ETH between 2017 and 2022?
include_graph: True

[BM25] top 3 / 20
1. id=16f49b83f2f6f702c29efa8987cb10c0ce68b7c7_fixed_0 | bm25_score=21.0650, fused_score=0.0115
   eth zurich at wef 2017: international exchange the eth delegation is also using the world economic forum 2017 as an opportunity for exchange with the huge range of wef participants from around the world. there are numero...
2. id=8adacd9ad5e2ee98d5e97dfe98c7b2587a0694ba_fixed_0 | bm25_score=20.9375, fused_score=0.0113
   challenge the best in data science: eth zurich's continuing education offensive is gathering pace. after launching the [school for continuing education](/en/news-and-events/eth-news/news/2018/04/continuing-education.html...
3. id=1afe6e1de8c417b8ea5406f8d9b529ebf22966ef_fixed_0 | bm25_score=20.9113, fused_score=0.0111
   what previous bird flu outbreaks teach us: - the bird flu epidemic in china from 2013 to 2017 showed that pathogens can circulate in poultry farms for several

### Explanation: Debug helper cell

This cell is for diagnosis, not main pipeline logic.

Step-by-step:
1. Print top docs from each retriever (`BM25`, `Dense`, `GraphRAG`).
2. Run full pipeline on the same query.
3. Print fused and reranked docs with scores.
4. Print trace info (`query_type`, `query_hints`, `critic_feedback`, `evidence_ids`).

Use this when answers are wrong to find exactly which stage fails.

## Step 3 Implementation Plan (Strategy A: Voting)

This section is a guided build plan after Step 2.

Goal:
- Implement Voting orchestration cleanly.
- Evaluate with required metrics (quantitative, qualitative, efficiency).
- Keep structure modular so we can improve or swap parts later.

How to use this plan:
1. Read one markdown cell.
2. Implement only that part in the next code cell.
3. Run and verify before moving to the next step.
4. Keep changes small and traceable.

---
from IPython.display import display, HTML

### Overall implementation structure (Mermaid)

```mermaid
flowchart TD
    A[Step 2 Base Ready] --> B[Step 3 Config Cell]
    B --> C[Optional Bilingual Query Helper]
    C --> D[Voting Orchestrator Build]
    D --> E[Single Query Smoke Test]
    E --> F[Batch Quantitative Eval<br/>P@k / R@k / MRR / optional nDCG]
    F --> G[Efficiency Eval<br/>Latency + Retry Count + Cost Proxy]
    G --> H[Qualitative Eval<br/>Explainability + Complementarity + Failures]
    H --> I[Comparative Analysis vs Another Strategy]
    I --> J[Charts + Tables + Export CSV/JSON]
    J --> K[Checklist + Team Handoff]

    subgraph Voting Core Flow
        D1[Query Understanding] --> D2[Retrieve BM25 + Dense + Graph]
        D2 --> D3[Weighted RRF Fusion]
        D3 --> D4[ReRank]
        D4 --> D5[Synthesize]
        D5 --> D6[Critic]
        D6 -->|if fail| D7[One Retry: boost weights]
        D7 --> D2
        D6 -->|if pass| D8[Return + Trace]
    end
    D --> D1
```

### Environment & coding structure for next part

Recommended structure:
1. `Config` + shared imports (all thresholds and paths in one place).
2. `Helpers` (ID handling, safe unique, trace formatting).
3. `Voting orchestrator` (single entry function/class).
4. `Quick debug run` (one query with full trace).
5. `Batch evaluation` (P@k, Recall@k, MRR).
6. `Efficiency` (latency + rough compute cost proxy).
7. `Qualitative analysis` (explainability, complementarity, failures).
8. `Comparison tables/charts` (vs another strategy).

Why this order:
- Build first, validate fast, then evaluate deeply.
- Easy for team collaboration and branch merging.

### Step 1 — Planning config cell

What to implement in the next code cell:
- Define a small config block for Strategy A.
- Keep all tunable values together.

Essential config items:
- `RETRIEVE_K`, `TOP_K`, `K_RRF`
- Voting weights: `W_BM25`, `W_DENSE`, `W_GRAPH`
- Evaluation `K_VALUES` (e.g., 1,3,5,10)
- Flags: `USE_BILINGUAL_QUERY`, `USE_RETRY`
- Benchmark paths: `PATH_QRELS`, `PATH_QUERIES` (needed for evaluation steps)

Reason:
- Centralized tuning avoids hidden magic values and makes comparisons fair.
- Including benchmark paths here ensures evaluation cells can find qrels without hunting for paths.

### Step 2 — Optional bilingual query helper

What to implement in the next code cell:
- Add a helper that returns query variants:
  - English query (original)
  - Optional German translation (if enabled)

Reason:
- Corpus contains German content; query-language mismatch can hurt retrieval.
- Dual-query retrieval is simple and usually higher impact than adding a full translation agent now.

### Step 3 — Build Voting orchestrator

What to implement in the next code cell:
- A `voting_orchestrate(...)` function (or small class method) that does:
  1. Query understanding (optional dynamic hints).
  2. Retrieve from BM25 + Dense + Graph (for each query variant if bilingual enabled).
  3. Weighted RRF fusion.
  4. Re-ranking.
  5. Answer synthesis (extractive, from top reranked docs).
  6. Critic check + optional one retry (retry loops back to step 2).
  7. Return final answer + docs + trace dictionary.

Trace should include:
- weights used
- retrieval counts per agent
- retry triggered or not
- final evidence IDs

Reason:
- This is your Strategy A core and also supports explainability requirements.

### Step 4 — Quick smoke test (single query)

What to implement in the next code cell:
- Run one sample query through Strategy A.
- Print:
  - query type
  - weights
  - top IDs
  - critic status
  - final answer

Reason:
- Catch pipeline issues early before expensive batch evaluation.

### Step 5 — Quantitative evaluation loop

What to implement in the next code cell:
- Loop over benchmark QA queries.
- Build run dict for Strategy A.
- Compute and report:
  - `Precision@k`
  - `Recall@k`
  - `MRR`
  - optional `nDCG@k` (recommended for ranking quality)

Output:
- Per-query table
- Summary table

Reason:
- These are mandatory Step 3 quantitative deliverables.
- `nDCG@k` gives extra evidence for ranking quality in report.

### Step 6 — Efficiency evaluation

What to implement in the next code cell:
- Time each query run.
- Report:
  - average latency
  - median latency
  - P95 latency
  - retries count (how often critic-triggered retry happens)
  - optional proxy for cost (e.g., model calls count)

Reason:
- Step 3 explicitly asks for system efficiency evidence.
- Retry frequency helps explain latency/cost behavior.

### Step 7 — Qualitative analysis block

What to implement in the next code cell:
- Explainability examples:
  - show strategy trace for selected queries.
- Complementarity analysis:
  - overlap between BM25/Dense/Graph result sets.
- Failure analysis:
  - list low-score queries and categorize likely causes.

Reason:
- This satisfies qualitative requirements and strengthens your final report discussion.

### Step 8 — Comparative analysis & visualizations

What to implement in the next code cell:
- Build comparison table vs at least one other orchestration strategy.
- Use the same QA set, same qrels, and same `K_VALUES` for fairness.
- Add charts:
  - MRR bar chart
  - P@k / R@k heatmap or grouped bars
  - latency comparison chart
- Optional innovation mini-study:
  - run 2–3 weight settings for Voting and report sensitivity/ablation.

Reason:
- Comparative analysis with visual evidence is required for deliverables.
- Fair setup + small ablation strengthens credibility and innovation.

### Step 9 — Completion checklist before sharing

- [ ] Notebook runs top-to-bottom without errors.
- [ ] Strategy A outputs docs + trace reliably.
- [ ] Quantitative metrics table saved.
- [ ] Efficiency metrics included (latency + retries count).
- [ ] Qualitative findings documented.
- [ ] Plots generated and readable.
- [ ] Artifacts exported (CSV/JSON) for reproducibility.
- [ ] Notes added for what to merge into team final notebook.

This cell helps collaboration and keeps branch handoff clean.